## I will create the model for monitoring the drift. 
It will monitor all the data in the streaming table
Another table will monitor data from the roilling average

## Step 1 — Configure data sources 

In [0]:
from pyspark.sql import functions as F

# Historical reference dataset
REFERENCE_TABLE = (
    "predictivemaintenancecatalog.gold_data.all_features"
)

# Incoming Gold data
CURRENT_TABLE = (
    "predictivemaintenancecatalog.gold_data.streaming"
)

# All 11 input features
DRIFT_FEATURES = [
    # Original features
    "Type",
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min",

    # Engineered features
    "Mechanical_power_W",
    "Temperature_difference_K",
    "Torque_wear",
    "Speed_wear",
    "Power_wear"
]

# Separate categorical and numerical features
CATEGORICAL_FEATURES = ["Type"]

NUMERIC_FEATURES = [
    c for c in DRIFT_FEATURES
    if c not in CATEGORICAL_FEATURES
]

# Load reference and incoming datasets
reference_df = (
    spark.table(REFERENCE_TABLE)
    .select(*DRIFT_FEATURES)
)

current_df = (
    spark.table(CURRENT_TABLE)
    .select(*DRIFT_FEATURES)
)

print("Reference records:", reference_df.count())
print("Incoming records:", current_df.count())

print("Total features:", len(DRIFT_FEATURES))
print("Numerical features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))

### Data quality check

In [0]:
def check_nulls(df, name):
    print(f"\n{name}")

    df.select([
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in DRIFT_FEATURES
    ]).show()

check_nulls(reference_df, "REFERENCE DATA")
check_nulls(current_df, "INCOMING DATA")

##Step 3 — Build the historical reference distributions

### Create reference distributions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType
)

DRIFT_REFERENCE_TABLE = (
    "predictivemaintenancecatalog.gold_data."
    "prediction_monitoring_drift_reference"
)

N_BINS = 10

reference_records = reference_df.count()

baseline_rows = []

# ----------------------------------
# 1. Numerical feature distributions
# ----------------------------------

for feature in NUMERIC_FEATURES:

    probabilities = [
        i / N_BINS for i in range(1, N_BINS)
    ]

    cuts = reference_df.approxQuantile(
        feature,
        probabilities,
        0.001
    )

    # Remove duplicate boundaries
    cuts = sorted(set(float(x) for x in cuts))

    # Assign observations to reference-defined bins
    bin_expr = F.lit(len(cuts)).cast("int")

    for i in reversed(range(len(cuts))):
        bin_expr = (
            F.when(
                F.col(feature) <= F.lit(cuts[i]),
                F.lit(i)
            )
            .otherwise(bin_expr)
        )

    counts = (
        reference_df
        .select(bin_expr.alias("bin_index"))
        .groupBy("bin_index")
        .count()
        .collect()
    )

    count_dict = {
        int(row["bin_index"]): int(row["count"])
        for row in counts
    }

    for i in range(len(cuts) + 1):

        count = count_dict.get(i, 0)

        lower = (
            float("-inf") if i == 0
            else cuts[i - 1]
        )

        upper = (
            float("inf") if i == len(cuts)
            else cuts[i]
        )

        baseline_rows.append((
            feature,
            "numeric",
            str(i),
            count,
            count / reference_records,
            lower,
            upper
        ))


# ----------------------------------
# 2. Categorical feature distribution
# ----------------------------------

for feature in CATEGORICAL_FEATURES:

    counts = (
        reference_df
        .groupBy(feature)
        .count()
        .collect()
    )

    for row in counts:

        category = str(row[feature])
        count = int(row["count"])

        baseline_rows.append((
            feature,
            "categorical",
            category,
            count,
            count / reference_records,
            None,
            None
        ))

    # Reserve a category for previously unseen values
    baseline_rows.append((
        feature,
        "categorical",
        "__OTHER__",
        0,
        0.0,
        None,
        None
    ))


# ----------------------------------
# 3. Save reference distributions
# ----------------------------------

schema = StructType([
    StructField("feature", StringType(), False),
    StructField("feature_type", StringType(), False),
    StructField("bin", StringType(), False),
    StructField("reference_count", LongType(), False),
    StructField("reference_pct", DoubleType(), False),
    StructField("lower_bound", DoubleType(), True),
    StructField("upper_bound", DoubleType(), True)
])

if not spark.catalog.tableExists(DRIFT_REFERENCE_TABLE):

    baseline_df = spark.createDataFrame(
        baseline_rows,
        schema=schema
    )

    (
        baseline_df
        .withColumn(
            "reference_records",
            F.lit(reference_records).cast("long")
        )
        .withColumn(
            "created_at",
            F.current_timestamp()
        )
        .write
        .format("delta")
        .mode("error")
        .saveAsTable(DRIFT_REFERENCE_TABLE)
    )

    print("Historical reference distributions created.")

else:

    print("Reference table already exists. Keeping existing baseline.")

### Inspect the reference distributions

In [0]:
baseline_df = spark.table(DRIFT_REFERENCE_TABLE)

display(
    baseline_df
    .orderBy("feature", "bin")
)

## Step 4 — Calculate PSI and KS statistics
1. PSI (Population Stability Index): Measures how much the incoming feature distribution differs from the historical reference.
2. KS statistic (Kolmogorov-Smirnov): Measures the maximum difference between the two empirical cumulative distributions.
3. KS p-value: Helps assess whether the observed distribution difference is statistically significant.

### Calculate drift metrics for test data(all data that is aleady in the streaming data)

In [0]:
import numpy as np
from scipy.stats import ks_2samp
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Load our fixed historical reference
baseline_df = spark.table(DRIFT_REFERENCE_TABLE)

EPSILON = 1e-6

# PSI calculation with smoothing for zero-frequency bins
def calculate_psi(reference_pct, current_pct):

    reference_pct = np.array(reference_pct, dtype=float)
    current_pct = np.array(current_pct, dtype=float)

    # Smooth zero proportions and renormalize
    reference_pct = (
        reference_pct + EPSILON
    ) / (
        reference_pct.sum() + EPSILON * len(reference_pct)
    )

    current_pct = (
        current_pct + EPSILON
    ) / (
        current_pct.sum() + EPSILON * len(current_pct)
    )

    psi = np.sum(
        (current_pct - reference_pct)
        * np.log(current_pct / reference_pct)
    )

    return float(psi)


# Collect numerical values for KS tests
reference_values = (
    reference_df.select(*NUMERIC_FEATURES).toPandas()
)

current_values = (
    current_df.select(*NUMERIC_FEATURES).toPandas()
)

drift_results = []

# ----------------------------------
# Numerical features: PSI and KS
# ----------------------------------

for feature in NUMERIC_FEATURES:

    baseline_rows = (
        baseline_df
        .filter(F.col("feature") == feature)
        .orderBy(F.col("bin").cast("int"))
        .collect()
    )

    # Original reference bin boundaries
    cuts = [
        float(row["upper_bound"])
        for row in baseline_rows[:-1]
    ]

    # Assign current observations to historical bins
    bin_expr = F.lit(len(cuts)).cast("int")

    for i in reversed(range(len(cuts))):
        bin_expr = (
            F.when(
                F.col(feature) <= F.lit(cuts[i]),
                F.lit(i)
            )
            .otherwise(bin_expr)
        )

    current_counts = (
        current_df
        .select(bin_expr.alias("bin_index"))
        .groupBy("bin_index")
        .count()
        .collect()
    )

    count_dict = {
        int(row["bin_index"]): int(row["count"])
        for row in current_counts
    }

    reference_pct = [
        float(row["reference_pct"])
        for row in baseline_rows
    ]

    current_total = len(current_values)

    current_pct = [
        count_dict.get(i, 0) / current_total
        for i in range(len(baseline_rows))
    ]

    psi = calculate_psi(reference_pct, current_pct)

    # KS two-sample test
    ks_result = ks_2samp(
        reference_values[feature].to_numpy(),
        current_values[feature].to_numpy()
    )

    drift_results.append((
        feature,
        "numeric",
        current_total,
        psi,
        float(ks_result.statistic),
        float(ks_result.pvalue)
    ))


# ----------------------------------
# Categorical feature: PSI only
# ----------------------------------

for feature in CATEGORICAL_FEATURES:

    baseline_rows = (
        baseline_df
        .filter(F.col("feature") == feature)
        .collect()
    )

    known_categories = {
        row["bin"]
        for row in baseline_rows
        if row["bin"] != "__OTHER__"
    }

    current_counts = (
        current_df
        .groupBy(feature)
        .count()
        .collect()
    )

    category_counts = {
        str(row[feature]): int(row["count"])
        for row in current_counts
    }

    current_total = sum(category_counts.values())

    reference_pct = []
    current_pct = []

    for row in baseline_rows:

        category = row["bin"]

        reference_pct.append(
            float(row["reference_pct"])
        )

        if category == "__OTHER__":
            count = sum(
                value
                for key, value in category_counts.items()
                if key not in known_categories
            )
        else:
            count = category_counts.get(category, 0)

        current_pct.append(count / current_total)

    psi = calculate_psi(reference_pct, current_pct)

    drift_results.append((
        feature,
        "categorical",
        current_total,
        psi,
        None,
        None
    ))


# ----------------------------------
# Create results DataFrame
# ----------------------------------

result_schema = StructType([
    StructField("feature", StringType(), False),
    StructField("feature_type", StringType(), False),
    StructField("current_records", LongType(), False),
    StructField("psi", DoubleType(), False),
    StructField("ks_statistic", DoubleType(), True),
    StructField("ks_pvalue", DoubleType(), True)
])

drift_metrics_df = (
    spark.createDataFrame(drift_results, result_schema)
    .withColumn(
        "psi_status",
        F.when(F.col("psi") < 0.10, "STABLE")
        .when(F.col("psi") < 0.25, "MODERATE_DRIFT")
        .otherwise("SIGNIFICANT_DRIFT")
    )
    .withColumn(
        "ks_significant",
        F.when(
            F.col("ks_pvalue").isNotNull(),
            F.col("ks_pvalue") < 0.05
        )
    )
    .withColumn(
        "calculated_at",
        F.current_timestamp()
    )
)

display(
    drift_metrics_df.orderBy(F.desc("psi"))
)

### PSI visualization

In [0]:
import matplotlib.pyplot as plt

# Convert the Spark results to Pandas
psi_plot_df = (
    drift_metrics_df
    .select("feature", "psi")
    .orderBy(F.desc("psi"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    psi_plot_df["feature"],
    psi_plot_df["psi"]
)

# Reverse order so highest PSI appears at the top
ax.invert_yaxis()

# PSI moderate drift threshold
ax.axvline(
    0.10,
    linestyle="--",
    label="Moderate drift threshold (0.10)",
    color="red"
)
# Dynamically determine x-axis limit
psi_threshold = 0.10
max_psi = psi_plot_df["psi"].max()

x_max = max(max_psi, psi_threshold) * 1.15

ax.set_xlim(0, x_max)

ax.set_xlabel("Population Stability Index (PSI)")
ax.set_ylabel("Feature")


# Add PSI values beside bars
for bar, value in zip(bars, psi_plot_df["psi"]):

    ax.text(
        value + (x_max * 0.008),
        bar.get_y() + bar.get_height() / 2,
        f"{value:.4f}",
        va="center"
    )

ax.set_title(
    "Feature Drift Monitoring — PSI",
    fontsize=14
)

ax.legend(loc="upper right")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

### Historical versus incoming distributions Distribution comparison

In [0]:
import numpy as np
import matplotlib.pyplot as plt

# Load the saved reference distributions
baseline_pdf = (
    spark.table(DRIFT_REFERENCE_TABLE)
    .toPandas()
)

# Load numerical observations
reference_pdf = reference_df.select(
    *NUMERIC_FEATURES
).toPandas()

current_pdf = current_df.select(
    *NUMERIC_FEATURES
).toPandas()

# Create 5 x 2 subplot layout
fig, axes = plt.subplots(
    5, 2,
    figsize=(16, 20)
)

axes = axes.flatten()

for i, feature in enumerate(NUMERIC_FEATURES):

    ax = axes[i]

    # Historical reference bins
    feature_bins = (
        baseline_pdf[
            baseline_pdf["feature"] == feature
        ]
        .copy()
        .sort_values("bin", key=lambda x: x.astype(int))
    )

    # Use historical bin boundaries
    cuts = (
        feature_bins["upper_bound"]
        .iloc[:-1]
        .to_numpy(dtype=float)
    )

    reference_pct = (
        feature_bins["reference_pct"]
        .to_numpy()
    )

    # Assign incoming observations to historical bins
    values = current_pdf[feature].dropna().to_numpy()

    bin_indices = np.searchsorted(
        cuts,
        values,
        side="left"
    )

    current_counts = np.bincount(
        bin_indices,
        minlength=len(feature_bins)
    )

    current_pct = current_counts / len(values)

    # Grouped bar chart
    x = np.arange(len(feature_bins))
    width = 0.38

    ax.bar(
        x - width / 2,
        reference_pct * 100,
        width,
        label="Historical"
    )

    ax.bar(
        x + width / 2,
        current_pct * 100,
        width,
        label="Incoming"
    )

    ax.set_title(feature)
    ax.set_xlabel("Historical reference bins")
    ax.set_ylabel("Observations (%)")

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"B{j+1}" for j in x]
    )

    ax.grid(axis="y", alpha=0.25)

    if i == 0:
        ax.legend()

fig.suptitle(
    "Historical vs Incoming Feature Distributions",
    fontsize=16,
    y=1.01
)

plt.tight_layout()
plt.show()

### Historical vs Incoming Box Plots

In [0]:
import matplotlib.pyplot as plt

# Load numerical features
reference_pdf = reference_df.select(
    *NUMERIC_FEATURES
).toPandas()

current_pdf = current_df.select(
    *NUMERIC_FEATURES
).toPandas()

# Create 5 rows x 2 columns
fig, axes = plt.subplots(
    5, 2,
    figsize=(15, 20)
)

axes = axes.flatten()

for i, feature in enumerate(NUMERIC_FEATURES):

    ax = axes[i]

    # Historical and incoming distributions
    data = [
        reference_pdf[feature].dropna(),
        current_pdf[feature].dropna()
    ]

    ax.boxplot(
        data,
        labels=["Historical", "Incoming"],
        showfliers=False
    )

    ax.set_title(feature)
    ax.set_ylabel("Value")

    ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "Historical vs Incoming Feature Distributions — Box Plots",
    fontsize=16,
    y=1.01
)

plt.tight_layout()
plt.show()

### Historical vs Incoming Machine Type for Cat data

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Calculate historical category counts
reference_type = (
    reference_df
    .groupBy("Type")
    .count()
    .toPandas()
)

# Calculate incoming category counts
current_type = (
    current_df
    .groupBy("Type")
    .count()
    .toPandas()
)

# Combine both datasets
type_plot_df = (
    reference_type
    .rename(columns={"count": "Historical"})
    .merge(
        current_type.rename(
            columns={"count": "Incoming"}
        ),
        on="Type",
        how="outer"
    )
    .fillna(0)
)

# Sort categories: L, M, H, then any unexpected categories
category_order = ["L", "M", "H"]

type_plot_df["Type"] = type_plot_df["Type"].astype(str)

type_plot_df["_order"] = type_plot_df["Type"].apply(
    lambda x: (
        category_order.index(x)
        if x in category_order
        else len(category_order)
    )
)

type_plot_df = (
    type_plot_df
    .sort_values(["_order", "Type"])
    .drop(columns="_order")
)

# Convert counts to percentages
type_plot_df["Historical_pct"] = (
    type_plot_df["Historical"]
    / type_plot_df["Historical"].sum() * 100
)

type_plot_df["Incoming_pct"] = (
    type_plot_df["Incoming"]
    / type_plot_df["Incoming"].sum() * 100
)

# Plot
x = np.arange(len(type_plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

bars1 = ax.bar(
    x - width / 2,
    type_plot_df["Historical_pct"],
    width,
    label="Historical"
)

bars2 = ax.bar(
    x + width / 2,
    type_plot_df["Incoming_pct"],
    width,
    label="Incoming"
)

# Add percentage labels
ax.bar_label(bars1, fmt="%.2f%%", padding=3)
ax.bar_label(bars2, fmt="%.2f%%", padding=3)

ax.set_xticks(x)
ax.set_xticklabels(type_plot_df["Type"])

ax.set_xlabel("Machine Type")
ax.set_ylabel("Observations (%)")

ax.set_title(
    "Historical vs Incoming Machine Type Distribution"
)

ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)

ax.set_ylim(
    0,
    max(
        type_plot_df["Historical_pct"].max(),
        type_plot_df["Incoming_pct"].max()
    ) * 1.2
)

plt.tight_layout()
plt.show()

### Save drift monitoring results

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

DRIFT_RESULTS_TABLE = (
    "predictivemaintenancecatalog.gold_data."
    "prediction_monitoring_drift"
)

# Get information about the current Gold dataset
current_summary = (
    spark.table(CURRENT_TABLE)
    .agg(
        F.count("*").alias("current_records"),
        F.min("_gold_processed_at").alias("window_start"),
        F.max("_gold_processed_at").alias("window_end")
    )
    .first()
)

reference_records = reference_df.count()

# Create a repeatable ID for the same dataset snapshot
run_id = (
    f"{current_summary['current_records']}_"
    f"{current_summary['window_start']}_"
    f"{current_summary['window_end']}"
)

# Add monitoring metadata
drift_snapshot_df = (
    drift_metrics_df
    .withColumn("monitoring_run_id", F.lit(run_id))
    .withColumn("reference_table", F.lit(REFERENCE_TABLE))
    .withColumn("current_table", F.lit(CURRENT_TABLE))
    .withColumn(
        "reference_records",
        F.lit(reference_records).cast("long")
    )
    .withColumn(
        "gold_processed_start",
        F.lit(current_summary["window_start"]).cast("timestamp")
    )
    .withColumn(
        "gold_processed_end",
        F.lit(current_summary["window_end"]).cast("timestamp")
    )
)

# Save results
if not spark.catalog.tableExists(DRIFT_RESULTS_TABLE):

    (
        drift_snapshot_df.write
        .format("delta")
        .mode("error")
        .saveAsTable(DRIFT_RESULTS_TABLE)
    )

    print("Drift monitoring table created.")

else:

    (
        DeltaTable.forName(spark, DRIFT_RESULTS_TABLE)
        .alias("target")
        .merge(
            drift_snapshot_df.alias("source"),
            """
            target.monitoring_run_id = source.monitoring_run_id
            AND target.feature = source.feature
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Drift monitoring results saved.")

print("Monitoring run:", run_id)

## Rolling Average Monitoring

In [0]:
### Extract the latest 1,000 observations

In [0]:
from pyspark.sql import functions as F

WINDOW_SIZE = 1000

# Read Gold streaming table
gold_stream_df = spark.table(CURRENT_TABLE)

# Extract the latest 1,000 observations
rolling_current_df = (
    gold_stream_df
    .select(
        "UDI",
        "_gold_processed_at",
        *DRIFT_FEATURES
    )
    .orderBy(
        F.col("_gold_processed_at").desc(),
        F.col("UDI").desc()
    )
    .limit(WINDOW_SIZE)
)

# Validate
window_records = rolling_current_df.count()

print("Rolling window records:", window_records)

if window_records != WINDOW_SIZE:
    raise ValueError(
        f"Expected {WINDOW_SIZE} records, found {window_records}"
    )

# Inspect window boundaries
rolling_current_df.agg(
    F.min("_gold_processed_at").alias("window_start"),
    F.max("_gold_processed_at").alias("window_end"),
    F.min("UDI").alias("minimum_udi"),
    F.max("UDI").alias("maximum_udi")
).show(truncate=False)

display(rolling_current_df.limit(10))

### Rolling PSI and KS
compare rolling average to the historical 8000 dataset

In [0]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Load fixed historical reference distributions
baseline_df = spark.table(DRIFT_REFERENCE_TABLE)

# Convert datasets to Pandas
reference_pdf = reference_df.select(
    *NUMERIC_FEATURES
).toPandas()

rolling_pdf = rolling_current_df.select(
    *DRIFT_FEATURES
).toPandas()

EPSILON = 1e-6


# ----------------------------------
# PSI calculation
# ----------------------------------

def calculate_psi(reference_pct, current_pct):

    reference_pct = np.array(reference_pct, dtype=float)
    current_pct = np.array(current_pct, dtype=float)

    reference_pct = (
        reference_pct + EPSILON
    ) / (
        reference_pct.sum() + EPSILON * len(reference_pct)
    )

    current_pct = (
        current_pct + EPSILON
    ) / (
        current_pct.sum() + EPSILON * len(current_pct)
    )

    return float(
        np.sum(
            (current_pct - reference_pct)
            * np.log(current_pct / reference_pct)
        )
    )


rolling_results = []


# ----------------------------------
# Numerical features: PSI + KS
# ----------------------------------

for feature in NUMERIC_FEATURES:

    baseline_rows = (
        baseline_df
        .filter(F.col("feature") == feature)
        .orderBy(F.col("bin").cast("int"))
        .collect()
    )

    # Historical bin boundaries
    cuts = [
        float(row["upper_bound"])
        for row in baseline_rows[:-1]
    ]

    reference_pct = [
        float(row["reference_pct"])
        for row in baseline_rows
    ]

    # Assign rolling observations to historical bins
    values = rolling_pdf[feature].dropna().to_numpy()

    bin_indices = np.searchsorted(
        cuts,
        values,
        side="left"
    )

    current_counts = np.bincount(
        bin_indices,
        minlength=len(baseline_rows)
    )

    current_pct = current_counts / len(values)

    # PSI
    psi = calculate_psi(
        reference_pct,
        current_pct
    )

    # KS two-sample test
    ks_result = ks_2samp(
        reference_pdf[feature].dropna(),
        values
    )

    rolling_results.append((
        feature,
        "numeric",
        len(values),
        psi,
        float(ks_result.statistic),
        float(ks_result.pvalue)
    ))


# ----------------------------------
# Categorical feature: PSI
# ----------------------------------

for feature in CATEGORICAL_FEATURES:

    baseline_rows = (
        baseline_df
        .filter(F.col("feature") == feature)
        .collect()
    )

    known_categories = {
        row["bin"]
        for row in baseline_rows
        if row["bin"] != "__OTHER__"
    }

    category_counts = (
        rolling_pdf[feature]
        .astype(str)
        .value_counts()
        .to_dict()
    )

    current_total = sum(category_counts.values())

    reference_pct = []
    current_pct = []

    for row in baseline_rows:

        category = row["bin"]

        reference_pct.append(
            float(row["reference_pct"])
        )

        if category == "__OTHER__":

            count = sum(
                value
                for key, value in category_counts.items()
                if key not in known_categories
            )

        else:
            count = category_counts.get(category, 0)

        current_pct.append(count / current_total)

    psi = calculate_psi(
        reference_pct,
        current_pct
    )

    rolling_results.append((
        feature,
        "categorical",
        current_total,
        psi,
        None,
        None
    ))


# ----------------------------------
# Create rolling results DataFrame
# ----------------------------------

result_schema = StructType([
    StructField("feature", StringType(), False),
    StructField("feature_type", StringType(), False),
    StructField("current_records", LongType(), False),
    StructField("psi", DoubleType(), False),
    StructField("ks_statistic", DoubleType(), True),
    StructField("ks_pvalue", DoubleType(), True)
])

rolling_drift_metrics_df = (
    spark.createDataFrame(
        rolling_results,
        schema=result_schema
    )
    .withColumn(
        "psi_status",
        F.when(F.col("psi") < 0.10, "STABLE")
        .when(F.col("psi") < 0.25, "MODERATE_DRIFT")
        .otherwise("SIGNIFICANT_DRIFT")
    )
    .withColumn(
        "ks_significant",
        F.when(
            F.col("ks_pvalue").isNotNull(),
            F.col("ks_pvalue") < 0.05
        )
    )
    .withColumn(
        "calculated_at",
        F.current_timestamp()
    )
)

display(
    rolling_drift_metrics_df
    .orderBy(F.desc("psi"))
)

### Rolling PSI bar chart

In [0]:
import matplotlib.pyplot as plt

psi_rolling_pdf = (
    rolling_drift_metrics_df
    .select("feature", "psi")
    .orderBy(F.desc("psi"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    psi_rolling_pdf["feature"],
    psi_rolling_pdf["psi"]
)

ax.invert_yaxis()

# Drift threshold
psi_threshold = 0.10
max_psi = psi_rolling_pdf["psi"].max()

x_max = max(max_psi, psi_threshold) * 1.15

ax.axvline(
    psi_threshold,
    linestyle="--",
    label="Moderate drift threshold (0.10)",
    color ="red"
)

# Dynamic labels
for bar, value in zip(bars, psi_rolling_pdf["psi"]):

    ax.text(
        value + (x_max * 0.008),
        bar.get_y() + bar.get_height() / 2,
        f"{value:.4f}",
        va="center"
    )

ax.set_xlim(0, x_max)

ax.set_xlabel("Population Stability Index (PSI)")
ax.set_ylabel("Feature")

ax.set_title(
    "Rolling Feature Drift Monitoring — PSI",
    fontsize=14
)

ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

### Historical vs Rolling Distribution

In [0]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for i, feature in enumerate(NUMERIC_FEATURES):

    ax = axes[i]

    # Retrieve historical reference bins
    baseline_rows = (
        baseline_df
        .filter(F.col("feature") == feature)
        .orderBy(F.col("bin").cast("int"))
        .collect()
    )

    cuts = [
        float(row["upper_bound"])
        for row in baseline_rows[:-1]
    ]

    historical_pct = np.array([
        float(row["reference_pct"]) * 100
        for row in baseline_rows
    ])

    # Assign rolling observations to historical bins
    values = rolling_pdf[feature].dropna().to_numpy()

    bin_indices = np.searchsorted(
        cuts, values, side="left"
    )

    rolling_counts = np.bincount(
        bin_indices,
        minlength=len(baseline_rows)
    )

    rolling_pct = (
        rolling_counts / len(values) * 100
    )

    # Plot
    x = np.arange(len(baseline_rows))
    width = 0.35

    ax.bar(
        x - width/2,
        historical_pct,
        width,
        label="Historical"
    )

    ax.bar(
        x + width/2,
        rolling_pct,
        width,
        label="Rolling"
    )

    ax.set_title(feature)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"B{j+1}" for j in range(len(baseline_rows))]
    )

    ax.set_xlabel("Historical reference bins")
    ax.set_ylabel("Observations (%)")
    ax.grid(axis="y", alpha=0.3)

    if i == 0:
        ax.legend()

fig.suptitle(
    "Historical vs Rolling Feature Distributions",
    fontsize=16,
    y=1.01
)

plt.tight_layout()
plt.show()

### Historical vs Rolling Box Plots

In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 2, figsize=(15, 20))
axes = axes.flatten()

for i, feature in enumerate(NUMERIC_FEATURES):

    ax = axes[i]

    data = [
        reference_pdf[feature].dropna(),
        rolling_pdf[feature].dropna()
    ]

    ax.boxplot(
        data,
        labels=["Historical", "Rolling"],
        showfliers=False
    )

    ax.set_title(feature)
    ax.set_ylabel("Value")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "Historical vs Rolling Feature Distributions — Box Plots",
    fontsize=16,
    y=1.01
)

plt.tight_layout()
plt.show()

### Historical vs Rolling Machine Type

In [0]:
import numpy as np
import matplotlib.pyplot as plt

# Historical category counts
reference_type = (
    reference_df
    .groupBy("Type")
    .count()
    .toPandas()
)

# Rolling category counts
rolling_type = (
    rolling_current_df
    .groupBy("Type")
    .count()
    .toPandas()
)

# Merge
type_plot_df = (
    reference_type
    .rename(columns={"count": "Historical"})
    .merge(
        rolling_type.rename(columns={"count": "Rolling"}),
        on="Type",
        how="outer"
    )
    .fillna(0)
)

# Sort categories
category_order = ["L", "M", "H"]

type_plot_df["Type"] = type_plot_df["Type"].astype(str)

type_plot_df["_order"] = type_plot_df["Type"].apply(
    lambda x: category_order.index(x)
    if x in category_order else len(category_order)
)

type_plot_df = (
    type_plot_df
    .sort_values(["_order", "Type"])
    .drop(columns="_order")
)

# Convert to percentages
type_plot_df["Historical_pct"] = (
    type_plot_df["Historical"]
    / type_plot_df["Historical"].sum() * 100
)

type_plot_df["Rolling_pct"] = (
    type_plot_df["Rolling"]
    / type_plot_df["Rolling"].sum() * 100
)

# Plot
x = np.arange(len(type_plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

bars1 = ax.bar(
    x - width/2,
    type_plot_df["Historical_pct"],
    width,
    label="Historical"
)

bars2 = ax.bar(
    x + width/2,
    type_plot_df["Rolling_pct"],
    width,
    label="Rolling"
)

ax.bar_label(bars1, fmt="%.2f%%", padding=3)
ax.bar_label(bars2, fmt="%.2f%%", padding=3)

ax.set_xticks(x)
ax.set_xticklabels(type_plot_df["Type"])

ax.set_xlabel("Machine Type")
ax.set_ylabel("Observations (%)")

ax.set_title(
    "Historical vs Rolling Machine Type Distribution"
)

ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)

ax.set_ylim(
    0,
    max(
        type_plot_df["Historical_pct"].max(),
        type_plot_df["Rolling_pct"].max()
    ) * 1.2
)

plt.tight_layout()
plt.show()

## save the data to a table called rolling_drift_metrics_df

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Rolling drift results table
ROLLING_DRIFT_RESULTS_TABLE = (
    "predictivemaintenancecatalog.gold_data."
    "prediction_monitoring_drift_rolling"
)

# ----------------------------------
# 1. Capture rolling window metadata
# ----------------------------------

rolling_summary = (
    rolling_current_df
    .agg(
        F.count("*").alias("rolling_records"),
        F.min("_gold_processed_at").alias("window_start"),
        F.max("_gold_processed_at").alias("window_end"),
        F.min("UDI").cast("string").alias("minimum_udi"),
        F.max("UDI").cast("string").alias("maximum_udi")
    )
    .first()
)

reference_records = reference_df.count()

# Create a repeatable ID for this rolling window
rolling_run_id = (
    f"{rolling_summary['rolling_records']}_"
    f"{rolling_summary['window_start']}_"
    f"{rolling_summary['window_end']}_"
    f"{rolling_summary['minimum_udi']}_"
    f"{rolling_summary['maximum_udi']}"
)

# ----------------------------------
# 2. Add monitoring metadata
# ----------------------------------

rolling_drift_snapshot_df = (
    rolling_drift_metrics_df

    .withColumn(
        "monitoring_run_id",
        F.lit(rolling_run_id)
    )

    .withColumn(
        "reference_table",
        F.lit(REFERENCE_TABLE)
    )

    .withColumn(
        "current_table",
        F.lit(CURRENT_TABLE)
    )

    .withColumn(
        "reference_records",
        F.lit(reference_records).cast("long")
    )

    .withColumn(
        "rolling_window_size",
        F.lit(rolling_summary["rolling_records"]).cast("long")
    )

    .withColumn(
        "rolling_window_start",
        F.lit(rolling_summary["window_start"]).cast("timestamp")
    )

    .withColumn(
        "rolling_window_end",
        F.lit(rolling_summary["window_end"]).cast("timestamp")
    )

    .withColumn(
        "minimum_udi",
        F.lit(rolling_summary["minimum_udi"])
    )

    .withColumn(
        "maximum_udi",
        F.lit(rolling_summary["maximum_udi"])
    )
)

# ----------------------------------
# 3. Persist results to Delta
# ----------------------------------

if not spark.catalog.tableExists(ROLLING_DRIFT_RESULTS_TABLE):

    (
        rolling_drift_snapshot_df.write
        .format("delta")
        .mode("error")
        .saveAsTable(ROLLING_DRIFT_RESULTS_TABLE)
    )

    print("Rolling drift monitoring table created.")

else:

    (
        DeltaTable.forName(
            spark,
            ROLLING_DRIFT_RESULTS_TABLE
        )
        .alias("target")
        .merge(
            rolling_drift_snapshot_df.alias("source"),
            """
            target.monitoring_run_id = source.monitoring_run_id
            AND target.feature = source.feature
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Rolling drift results saved.")

print("Rolling monitoring run:", rolling_run_id)

# ----------------------------------
# 4. Verify saved results
# ----------------------------------

display(
    spark.table(ROLLING_DRIFT_RESULTS_TABLE)
    .filter(
        F.col("monitoring_run_id") == rolling_run_id
    )
    .orderBy("feature")
)


## Creating Alert for the dashboard

### Create the alert classification

In [0]:

from pyspark.sql import functions as F

# ----------------------------------
# 1. Classify drift severity
# ----------------------------------

drift_alert_evaluation_df = (
    rolling_drift_snapshot_df

    .withColumn(
        "alert_severity",
        F.when(
            F.col("psi_status") == "STABLE", "NORMAL"
        )
        .when(
            F.col("psi_status") == "MODERATE_DRIFT", "WARNING"
        )
        .when(
            F.col("psi_status") == "SIGNIFICANT_DRIFT", "HIGH"
        )
        .otherwise("UNCLASSIFIED")
    )

    # ----------------------------------
    # 2. Add KS statistical evidence
    # ----------------------------------

    .withColumn(
        "ks_evidence",
        F.when(
            F.col("ks_pvalue").isNull(), "NOT_APPLICABLE"
        )
        .when(
            F.col("ks_pvalue") < 0.05, "SIGNIFICANT"
        )
        .otherwise("NOT_SIGNIFICANT")
    )

    # ----------------------------------
    # 3. Recommended action
    # ----------------------------------

    .withColumn(
        "recommended_action",
        F.when(
            F.col("alert_severity") == "HIGH",
            "Investigate feature distribution, sensor/data pipeline, operating conditions and potential model impact."
        )
        .when(
            F.col("alert_severity") == "WARNING",
            "Review feature distribution and operating conditions. Monitor subsequent windows."
        )
        .when(
            F.col("alert_severity") == "NORMAL",
            "No action required. Continue monitoring."
        )
        .otherwise(
            "Review drift metrics and classification."
        )
    )

    .withColumn(
        "alert_evaluated_at",
        F.current_timestamp()
    )
)

# ----------------------------------
# 4. Display results
# ----------------------------------

display(
    drift_alert_evaluation_df
    .select(
        "feature",
        "psi",
        "psi_status",
        "ks_statistic",
        "ks_pvalue",
        "ks_evidence",
        "alert_severity",
        "recommended_action"
    )
    .orderBy(F.desc("psi"))
)


### Persist the alert evaluations
predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_alerts

We will use it as an alert evaluation history table, with one record per feature per monitoring run.

This is important because:
- can see which features experienced drift.
- can compare drift across successive monitoring runs.
- can identify when a feature returned to normal.
- have an audit trail for our monitoring system.
We will not overwrite your existing rolling drift metrics table.

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ----------------------------------
# 1. Define alert history table
# ----------------------------------

DRIFT_ALERTS_TABLE = (
    "predictivemaintenancecatalog.gold_data."
    "prediction_monitoring_drift_alerts"
)

# ----------------------------------
# 2. Prepare alert history records
# ----------------------------------

alert_history_df = (
    drift_alert_evaluation_df

    .withColumn(
        "alert_status",
        F.when(
            F.col("alert_severity") == "NORMAL", "NO_ALERT"
        )
        .when(
            F.col("alert_severity").isin("WARNING", "HIGH"), "OPEN"
        )
        .otherwise("REVIEW")
    )

    .select(
        "monitoring_run_id",
        "feature",
        "feature_type",
        "current_records",
        "psi",
        "psi_status",
        "ks_statistic",
        "ks_pvalue",
        "ks_significant",
        "ks_evidence",
        "alert_severity",
        "alert_status",
        "recommended_action",
        "calculated_at",
        "alert_evaluated_at",
        "reference_table",
        "current_table",
        "reference_records",
        "rolling_window_size",
        "rolling_window_start",
        "rolling_window_end",
        "minimum_udi",
        "maximum_udi"
    )
)

# ----------------------------------
# 3. Create or update Delta table
# ----------------------------------

if not spark.catalog.tableExists(DRIFT_ALERTS_TABLE):

    (
        alert_history_df.write
        .format("delta")
        .mode("error")
        .saveAsTable(DRIFT_ALERTS_TABLE)
    )

    print("Drift alert history table created.")

else:

    (
        DeltaTable.forName(
            spark,
            DRIFT_ALERTS_TABLE
        )
        .alias("target")
        .merge(
            alert_history_df.alias("source"),
            """
            target.monitoring_run_id = source.monitoring_run_id
            AND target.feature = source.feature
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Drift alert history updated.")

# ----------------------------------
# 4. Verify current monitoring run
# ----------------------------------

current_alerts_df = (
    spark.table(DRIFT_ALERTS_TABLE)
    .filter(
        F.col("monitoring_run_id") == rolling_run_id
    )
)

print("Alert records:", current_alerts_df.count())

display(
    current_alerts_df
    .select(
        "feature",
        "psi",
        "alert_severity",
        "alert_status",
        "recommended_action"
    )
    .orderBy(F.desc("psi"))
)


### Alert lifecycle management

Before sending emails, we need to solve an important operational problem: how do we know whether an alert is new, continuing, escalating, or resolved?

I recommend keeping your existing prediction_monitoring_drift_alerts table as the per-run evaluation history.

We will create a separate table for the current state of each feature:

predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_alert_state
| Situation | State transition | Notification |
| :--- | :--- | :--- |
| New HIGH or WARNING | NEW → OPEN | Send |
| Same drift on next run | OPEN → OPEN | No |
| WARNING becomes HIGH | ESCALATED | Send |
| HIGH becomes WARNING | DOWNGRADED | No |
| First normal window | RECOVERY_PENDING | No |
| Two consecutive normal windows | RESOLVED | Send recovery email |

#### Initialize the current alert state


In [0]:

from pyspark.sql import functions as F

DRIFT_ALERT_STATE_TABLE = (
    "predictivemaintenancecatalog.gold_data."
    "prediction_monitoring_drift_alert_state"
)

# Get the current monitoring run's alert evaluations
current_alerts_df = (
    spark.table(DRIFT_ALERTS_TABLE)
    .filter(
        F.col("monitoring_run_id") == rolling_run_id
    )
)

# Check that records exist
if current_alerts_df.limit(1).count() == 0:
    raise ValueError("No alert evaluations found for the current monitoring run.")

# Initialize one state record per feature
initial_alert_state_df = (
    current_alerts_df

    .select(
        "feature",
        "feature_type",

        F.col("alert_severity").alias("current_severity"),

        "alert_status",

        F.when(
            F.col("alert_severity").isin("HIGH", "WARNING"),
            F.col("calculated_at")
        ).alias("first_detected_at"),

        F.when(
            F.col("alert_severity").isin("HIGH", "WARNING"),
            F.col("calculated_at")
        ).alias("last_detected_at"),

        F.lit(0).cast("int").alias(
            "consecutive_normal_windows"
        ),

        F.col("monitoring_run_id").alias(
            "last_processed_monitoring_run_id"
        ),

        F.col("calculated_at").alias("last_calculated_at"),

        F.when(
            F.col("alert_severity").isin("HIGH", "WARNING"),
            "NEW_ALERT"
        ).otherwise("NO_ALERT").alias("last_transition"),

        F.col("psi").alias("last_psi"),

        F.col("ks_pvalue").alias("last_ks_pvalue"),

        "recommended_action"
    )

    .withColumn(
        "last_notified_at",
        F.lit(None).cast("timestamp")
    )

    .withColumn(
        "last_updated_at",
        F.current_timestamp()
    )
)

# Create the state table only once.
# Never overwrite an existing alert state table.

if not spark.catalog.tableExists(DRIFT_ALERT_STATE_TABLE):

    (
        initial_alert_state_df.write
        .format("delta")
        .mode("error")
        .saveAsTable(DRIFT_ALERT_STATE_TABLE)
    )

    print("Alert state table created.")

else:

    print("Alert state table already exists. Existing states preserved.")

# Verify
display(
    spark.table(DRIFT_ALERT_STATE_TABLE)
    .orderBy(F.desc("last_psi"))
)


#### Process subsequent monitoring  runs and Implement state transitions


Here is the behavior we want:

| Previous state | New result | Transition |
| :--- | :--- | :--- |
| WARNING | HIGH | ESCALATED |
| HIGH | HIGH | CONTINUING |
| HIGH | WARNING | DOWNGRADED |
| HIGH | NORMAL (first window) | RECOVERY_PENDING |
| RECOVERY_PENDING | NORMAL (second window) | RESOLVED |
| RECOVERY_PENDING | HIGH | REOPENED |
| RESOLVED | HIGH | NEW_ALERT |

This code will:
- Read the latest monitoring run.
- Compare it with the existing alert state.
- Identify new, continuing, escalating and resolved alerts.
- Update the state table.
- Prevent processing the same monitoring run twice.

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ----------------------------------
# 1. Read current monitoring results
# ----------------------------------

current_run_df = (
    spark.table(DRIFT_ALERTS_TABLE)
    .filter(F.col("monitoring_run_id") == rolling_run_id)
)

if current_run_df.limit(1).count() == 0:
    raise ValueError("No records found for current monitoring run.")

state_df = spark.table(DRIFT_ALERT_STATE_TABLE)

# ----------------------------------
# 2. Identify unprocessed features
# ----------------------------------

pending_run_df = (
    current_run_df.alias("c")

    .join(
        state_df.select(
            "feature",
            "last_processed_monitoring_run_id"
        ).alias("s"),
        F.col("c.feature") == F.col("s.feature"),
        "left"
    )

    .filter(
        F.col("s.last_processed_monitoring_run_id").isNull()
        |
        (
            F.col("s.last_processed_monitoring_run_id")
            != F.col("c.monitoring_run_id")
        )
    )

    .select("c.*")
)

pending_count = pending_run_df.count()

if pending_count == 0:

    print(
        "This monitoring run has already been processed. "
        "No state changes required."
    )

else:

    # ----------------------------------
    # 3. Join new results to existing state
    # ----------------------------------

    joined_df = (
        pending_run_df.alias("c")
        .join(
            state_df.alias("s"),
            F.col("c.feature") == F.col("s.feature"),
            "left"
        )
    )

    has_previous = F.col("s.feature").isNotNull()

    active = F.col("c.alert_severity").isin(
        "HIGH", "WARNING"
    )

    previous_status = F.col("s.alert_status")
    previous_severity = F.col("s.current_severity")

    new_severity = F.col("c.alert_severity")

    normal_count = F.coalesce(
        F.col("s.consecutive_normal_windows"),
        F.lit(0)
    )

    next_normal_count = normal_count + 1

    # ----------------------------------
    # 4. Determine state transition
    # ----------------------------------

    transition = (
        F.when(
            (~has_previous) & active,
            "NEW_ALERT"
        )

        .when(
            ~has_previous,
            "NO_ALERT"
        )

        .when(
            (previous_status == "RECOVERY_PENDING") & active,
            "REOPENED"
        )

        .when(
            previous_status.isin("RESOLVED", "NO_ALERT") & active,
            "NEW_ALERT"
        )

        .when(
            (previous_status == "OPEN")
            & (previous_severity == "WARNING")
            & (new_severity == "HIGH"),
            "ESCALATED"
        )

        .when(
            (previous_status == "OPEN")
            & (previous_severity == "HIGH")
            & (new_severity == "WARNING"),
            "DOWNGRADED"
        )

        .when(
            active,
            "CONTINUING"
        )

        .when(
            previous_status.isin("OPEN", "RECOVERY_PENDING")
            & (next_normal_count < 2),
            "RECOVERY_PENDING"
        )

        .when(
            previous_status.isin("OPEN", "RECOVERY_PENDING")
            & (next_normal_count >= 2),
            "RESOLVED"
        )

        .when(
            previous_status == "RESOLVED",
            "RESOLVED"
        )

        .otherwise("NO_ALERT")
    )

    # ----------------------------------
    # 5. Build updated state records
    # ----------------------------------

    new_status = (
        F.when(
            transition.isin(
                "NEW_ALERT",
                "ESCALATED",
                "DOWNGRADED",
                "CONTINUING",
                "REOPENED"
            ),
            "OPEN"
        )
        .when(
            transition == "RECOVERY_PENDING",
            "RECOVERY_PENDING"
        )
        .when(
            transition == "RESOLVED",
            "RESOLVED"
        )
        .otherwise("NO_ALERT")
    )

    lifecycle_updates_df = joined_df.select(

        F.col("c.feature").alias("feature"),

        F.col("c.feature_type").alias("feature_type"),

        new_severity.alias("current_severity"),

        new_status.alias("alert_status"),

        F.when(
            transition == "NEW_ALERT",
            F.col("c.calculated_at")
        )
        .otherwise(F.col("s.first_detected_at"))
        .alias("first_detected_at"),

        F.when(
            active,
            F.col("c.calculated_at")
        )
        .otherwise(F.col("s.last_detected_at"))
        .alias("last_detected_at"),

        F.when(
            active,
            F.lit(0)
        )
        .when(
            transition.isin("RECOVERY_PENDING", "RESOLVED"),
            F.least(next_normal_count, F.lit(2))
        )
        .otherwise(normal_count)
        .cast("int")
        .alias("consecutive_normal_windows"),

        F.col("c.monitoring_run_id").alias(
            "last_processed_monitoring_run_id"
        ),

        F.col("c.calculated_at").alias(
            "last_calculated_at"
        ),

        transition.alias("last_transition"),

        F.col("c.psi").alias("last_psi"),

        F.col("c.ks_pvalue").alias("last_ks_pvalue"),

        F.col("c.recommended_action").alias(
            "recommended_action"
        ),

        F.col("s.last_notified_at").alias(
            "last_notified_at"
        ),

        F.current_timestamp().alias("last_updated_at")
    )

    # ----------------------------------
    # 6. Update persistent state table
    # ----------------------------------

    (
        DeltaTable.forName(
            spark,
            DRIFT_ALERT_STATE_TABLE
        )
        .alias("target")

        .merge(
            lifecycle_updates_df.alias("source"),
            "target.feature = source.feature"
        )

        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()

        .execute()
    )

    print(
        f"Processed {pending_count} feature records."
    )

    # ----------------------------------
    # 7. Inspect updated state
    # ----------------------------------

    display(
        spark.table(DRIFT_ALERT_STATE_TABLE)
        .orderBy(F.desc("last_psi"))
    )


#### Create alert event history

We currently have two tables:
1. prediction_monitoring_drift_alerts — historical drift evaluations.
2. prediction_monitoring_drift_alert_state — current state of each feature.
I want to introduce a third table:

predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_alert_events

This table records what happened during each monitoring run.

For example:
    
| Feature | Event | Notification |
| :--- | :--- | :--- |
| Temperature_difference_K | NEW_ALERT | Required |
| Tool_wear_min | NEW_ALERT | Required |
| Air_temperature_K | NEW_ALERT | Required |
| Temperature_difference_K | ESCALATED | Required |
| Temperature_difference_K | RECOVERY_PENDING | Not required |
| Temperature_difference_K | RESOLVED | Required |

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ----------------------------------
# 1. Define event history table
# ----------------------------------

DRIFT_ALERT_EVENTS_TABLE = (
    "predictivemaintenancecatalog.gold_data."
    "prediction_monitoring_drift_alert_events"
)

# ----------------------------------
# 2. Extract events from current state
# ----------------------------------

event_source_df = (
    spark.table(DRIFT_ALERT_STATE_TABLE)

    .filter(
        F.col("last_processed_monitoring_run_id") == rolling_run_id
    )

    # Ignore ordinary continuing states
    .filter(
        F.col("last_transition").isin(
            "NEW_ALERT",
            "ESCALATED",
            "DOWNGRADED",
            "REOPENED",
            "RECOVERY_PENDING",
            "RESOLVED"
        )
    )

    .withColumn(
        "event_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("last_processed_monitoring_run_id"),
                F.col("feature"),
                F.col("last_transition")
            ),
            256
        )
    )

    .select(
        "event_id",

        F.col("last_processed_monitoring_run_id")
        .alias("monitoring_run_id"),

        "feature",
        "feature_type",

        F.col("last_transition").alias("event_type"),

        F.col("current_severity").alias("severity"),

        "alert_status",

        F.col("last_psi").alias("psi"),

        F.col("last_ks_pvalue").alias("ks_pvalue"),

        "recommended_action",

        F.when(
            F.col("last_transition").isin(
                "NEW_ALERT",
                "ESCALATED",
                "REOPENED",
                "RESOLVED"
            ),
            True
        ).otherwise(False).alias("notification_required"),

        F.col("last_calculated_at").alias("event_timestamp"),

        F.current_timestamp().alias("recorded_at")
    )

    .withColumn(
        "notification_status",
        F.when(
            F.col("notification_required"),
            "PENDING"
        ).otherwise("NOT_REQUIRED")
    )
)

# ----------------------------------
# 3. Create or append event history
# ----------------------------------

if not spark.catalog.tableExists(DRIFT_ALERT_EVENTS_TABLE):

    (
        event_source_df.write
        .format("delta")
        .mode("error")
        .saveAsTable(DRIFT_ALERT_EVENTS_TABLE)
    )

    print("Alert event history table created.")

else:

    (
        DeltaTable.forName(
            spark,
            DRIFT_ALERT_EVENTS_TABLE
        )
        .alias("target")

        .merge(
            event_source_df.alias("source"),
            "target.event_id = source.event_id"
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

    print("Alert events inserted without duplicates.")

# ----------------------------------
# 4. Verify event history
# ----------------------------------

display(
    spark.table(DRIFT_ALERT_EVENTS_TABLE)
    .filter(
        F.col("monitoring_run_id") == rolling_run_id
    )
    .orderBy("feature")
)


#### Configure email notifications
I recommend using Databricks SQL Alerts, rather than writing custom email-sending code inside your notebook.

Databricks supports SQL alerts that evaluate query results and send notifications when configured conditions are met. We can also use SQL Alert tasks in Lakeflow Jobs to integrate them into our monitoring workflow

##### 1. First, create the notification query

In [0]:
%sql

SELECT
    feature,
    severity,
    ROUND(psi, 4) AS psi,
    ks_pvalue,
    recommended_action,
    event_timestamp

FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_alert_events

WHERE notification_required = TRUE
  AND notification_status = 'PENDING'

ORDER BY
    CASE severity
        WHEN 'HIGH' THEN 1
        WHEN 'WARNING' THEN 2
        ELSE 3
    END,
    psi DESC;


##### 2. Create a Databricks SQL Alert
In your Databricks workspace:
1. Navigate to Alerts.
2. Select Create Alert.
3. Paste the SQL query above into the query editor.
4. Select an available SQL warehouse.
5. Configure the alert condition.
Use these settings:
| Setting | Value |
| :--- | :--- |
| Column | feature |
| Aggregation | COUNT |
| Condition | Greater than |
| Threshold | 0 |



This means that whenever pending notification events exist, the alert condition is triggered.

##### Update notification status

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ----------------------------------
# 1. Give the Databricks SQL alert time to notify first
# ----------------------------------
# The SQL alert (60_Predictive_Maintenance_Alert) evaluates every 10 minutes
# and only picks up events whose notification_status is still 'PENDING'.
# Retiring them within the same monitoring run starves the alert, so only
# retire events that have stayed PENDING for at least
# NOTIFY_GRACE_PERIOD_HOURS (about 6 alert evaluation cycles).

NOTIFY_GRACE_PERIOD_HOURS = 1

grace_cutoff = (
    F.current_timestamp()
    - F.expr(f"INTERVAL {NOTIFY_GRACE_PERIOD_HOURS} HOUR")
)

pending_events = (
    spark.table(DRIFT_ALERT_EVENTS_TABLE)
    .filter(
        (F.col("notification_required") == True)
        & (F.col("notification_status") == "PENDING")
        & (F.col("event_timestamp") < grace_cutoff)
    )
)

pending_count = pending_events.count()

print("Pending notifications past the alert grace period:", pending_count)

# ----------------------------------
# 2. Update event notification status
# ----------------------------------

if pending_count > 0:

    # Monitoring runs whose events are being retired now
    # (collected BEFORE the update, since pending_events is lazy)
    retired_run_ids = [
        row["monitoring_run_id"]
        for row in pending_events
        .select("monitoring_run_id")
        .distinct()
        .collect()
    ]

    DeltaTable.forName(
        spark,
        DRIFT_ALERT_EVENTS_TABLE
    ).update(
        condition=(
            (F.col("notification_required") == True)
            & (F.col("notification_status") == "PENDING")
            & (F.col("event_timestamp") < grace_cutoff)
        ),
        set={
            "notification_status": F.lit("SENT")
        }
    )

    # ----------------------------------
    # 3. Update alert state timestamps
    # ----------------------------------

    DeltaTable.forName(
        spark,
        DRIFT_ALERT_STATE_TABLE
    ).update(
        condition=(
            F.col("last_processed_monitoring_run_id").isin(retired_run_ids)
            & F.col("last_transition").isin(
                "NEW_ALERT",
                "ESCALATED",
                "REOPENED",
                "RESOLVED"
            )
        ),
        set={
            "last_notified_at": F.current_timestamp(),
            "last_updated_at": F.current_timestamp()
        }
    )

    print(
        f"Notification statuses updated for runs: {retired_run_ids}"
    )

else:

    print(
        "No notifications to retire yet - "
        "all pending events are still within the alert grace period."
    )

# ----------------------------------
# 4. Verify
# ----------------------------------

display(
    spark.table(DRIFT_ALERT_EVENTS_TABLE)
    .groupBy("notification_status")
    .count()
)
